# LangGraph: State Management

## Outline
* مفهوم State در LangGraph
* `AgentState` پیش‌فرض
* ساخت State سفارشی با `TypedDict`
* نوشتن و خواندن State از داخل Tool
* مثال کاربردی: سبد خرید هوشمند
* مشاهده State با `get_state`

In [35]:
# pip install langchain langgraph langchain-openai python-dotenv
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [37]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)
print("آماده!")

آماده!


## ۱. State چیست؟

در LangGraph هر agent یک **State** (وضعیت) دارد که در طول مکالمه ذخیره و آپدیت می‌شود.

```
State پیش‌فرض (AgentState) فقط شامل:
  ├── messages: list[BaseMessage]
  └── ...

State سفارشی می‌تواند هر چیزی داشته باشد:
  ├── messages: list[BaseMessage]
  ├── user_name: str
  ├── cart_items: list[dict]
  ├── total_price: float
  └── discount_applied: bool
```

## ۲. State پیش‌فرض — AgentState

In [41]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver

# ساده‌ترین حالت — بدون State سفارشی
agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "test-1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "سلام! اسمم علیرضا هست"}]},
    config=config
)
print(response["messages"][-1].content)

سلام علیرضا! چطور می‌توانم به شما کمک کنم؟


In [42]:
# مشاهده State
state = agent.get_state(config)
print("کلیدهای State:", list(state.values.keys()))
print(f"\nتعداد پیام‌ها: {len(state.values['messages'])}")
for msg in state.values['messages']:
    role = type(msg).__name__.replace('Message', '')
    print(f"  [{role}]: {msg.content[:60]}")

کلیدهای State: ['messages']

تعداد پیام‌ها: 2
  [Human]: سلام! اسمم علیرضا هست
  [AI]: سلام علیرضا! چطور می‌توانم به شما کمک کنم؟


## ۳. State سفارشی — اضافه کردن فیلدهای دلخواه

In [44]:
from langchain.agents import AgentState

# تعریف State سفارشی با ارث‌بری از AgentState
class UserProfileState(AgentState):
    user_name: str        # اسم کاربر
    city: str             # شهر کاربر
    preferred_language: str  # زبان ترجیحی

print("State سفارشی تعریف شد!")
print("فیلدهای اضافه شده:")
print("  - user_name: str")
print("  - city: str")
print("  - preferred_language: str")

State سفارشی تعریف شد!
فیلدهای اضافه شده:
  - user_name: str
  - city: str
  - preferred_language: str


In [45]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

# Tool که State را می‌نویسد
@tool
def save_user_name(name: str, runtime: ToolRuntime) -> Command:
    """اسم کاربر را در State ذخیره می‌کند. وقتی کاربر اسمش را گفت استفاده کن."""
    return Command(update={
        "user_name": name,
        "messages": [ToolMessage(
            content=f"اسم '{name}' ذخیره شد.",
            tool_call_id=runtime.tool_call_id
        )]
    })

@tool
def save_city(city: str, runtime: ToolRuntime) -> Command:
    """شهر کاربر را در State ذخیره می‌کند."""
    return Command(update={
        "city": city,
        "messages": [ToolMessage(
            content=f"شهر '{city}' ذخیره شد.",
            tool_call_id=runtime.tool_call_id
        )]
    })

# Tool که State را می‌خواند
@tool
def get_user_profile(dummy: str, runtime: ToolRuntime) -> str:
    """اطلاعات ذخیره شده کاربر را برمی‌گرداند."""
    name = runtime.state.get("user_name", "ناشناس")
    city = runtime.state.get("city", "نامشخص")
    return f"نام: {name}، شهر: {city}"

print("Toolها ساخته شدند!")

Toolها ساخته شدند!


In [50]:
# ساخت agent با State سفارشی
profile_agent = create_agent(
    model=llm,
    tools=[save_user_name, save_city, get_user_profile],
    state_schema=UserProfileState,
    checkpointer=InMemorySaver(),
    system_prompt="""شما یک دستیار فارسی‌زبان هستید. 
وقتی کاربر اسم یا شهرش را گفت، آن را ذخیره کنید.
اگر کاربر پروفایلش را خواست، از tool مناسب استفاده کنید."""
)

config = {"configurable": {"thread_id": "profile-session-1"}}
print("Agent آماده است!")

Agent آماده است!


In [52]:
# تست ۱: معرفی کاربر
resp = profile_agent.invoke(
    {"messages": [{"role": "user", "content": "سلام! اسمم علیرضا هست و از تهران هستم"}]},
    config=config
)
print(resp["messages"][-1].content)

سلام علیرضا! اسم و شهر شما با موفقیت ذخیره شد. اگر سوال یا درخواست دیگری دارید، خوشحال می‌شوم کمک کنم!


In [53]:
# تست ۲: خواندن اطلاعات ذخیره شده
resp = profile_agent.invoke(
    {"messages": [{"role": "user", "content": "پروفایل منو نشون بده"}]},
    config=config
)
print(resp["messages"][-1].content)

پروفایل شما به شرح زیر است:
- نام: علیرضا
- شهر: تهران

اگر سوال دیگری دارید یا به کمکی نیاز دارید، بفرمایید!


In [55]:
# مشاهده کامل State
state = profile_agent.get_state(config)
print("=== State فعلی ===")
print(f"user_name: {state.values.get('user_name', 'تنظیم نشده')}")
print(f"city: {state.values.get('city', 'تنظیم نشده')}")
print(f"تعداد پیام‌ها: {len(state.values['messages'])}")

=== State فعلی ===
user_name: علیرضا
city: تهران
تعداد پیام‌ها: 9


## ۴. مثال کاربردی: سبد خرید هوشمند

یک agent که State را برای مدیریت سبد خرید استفاده می‌کند:

In [59]:
from typing import Annotated
from langgraph.graph.message import add_messages

# State سبد خرید
class CartState(AgentState):
    cart_items: list         # آیتم‌های سبد
    total_price: float       # جمع کل
    customer_name: str       # اسم مشتری

# محصولات نمونه
PRODUCTS = {
    "لپ‌تاپ": 45_000_000,
    "موبایل": 25_000_000,
    "هدفون": 3_500_000,
    "کیبورد": 2_800_000,
    "ماوس": 1_200_000,
    "وب‌کم": 4_500_000,
}

@tool
def add_to_cart(product_name: str, runtime: ToolRuntime) -> Command:
    """یک محصول به سبد خرید اضافه می‌کند."""
    if product_name not in PRODUCTS:
        available = '، '.join(PRODUCTS.keys())
        return Command(update={"messages": [ToolMessage(
            content=f"محصول '{product_name}' یافت نشد. محصولات موجود: {available}",
            tool_call_id=runtime.tool_call_id
        )]})
    
    price = PRODUCTS[product_name]
    current_items = runtime.state.get("cart_items") or []
    current_total = runtime.state.get("total_price") or 0.0
    
    new_items = current_items + [{"name": product_name, "price": price}]
    new_total = current_total + price
    
    return Command(update={
        "cart_items": new_items,
        "total_price": new_total,
        "messages": [ToolMessage(
            content=f"{product_name} ({price:,} تومان) به سبد اضافه شد. جمع کل: {new_total:,.0f} تومان",
            tool_call_id=runtime.tool_call_id
        )]
    })

@tool
def show_cart(dummy: str, runtime: ToolRuntime) -> str:
    """محتوای سبد خرید را نمایش می‌دهد."""
    items = runtime.state.get("cart_items") or []
    total = runtime.state.get("total_price") or 0.0
    
    if not items:
        return "سبد خرید شما خالی است."
    
    lines = ["=== سبد خرید ==="]
    for i, item in enumerate(items, 1):
        lines.append(f"{i}. {item['name']}: {item['price']:,} تومان")
    lines.append(f"\nجمع کل: {total:,.0f} تومان")
    return "\n".join(lines)

@tool
def clear_cart(dummy: str, runtime: ToolRuntime) -> Command:
    """سبد خرید را خالی می‌کند."""
    return Command(update={
        "cart_items": [],
        "total_price": 0.0,
        "messages": [ToolMessage(
            content="سبد خرید خالی شد.",
            tool_call_id=runtime.tool_call_id
        )]
    })

print("Toolهای سبد خرید آماده است!")

Toolهای سبد خرید آماده است!


In [61]:
# ساخت agent سبد خرید
cart_agent = create_agent(
    model=llm,
    tools=[add_to_cart, show_cart, clear_cart],
    state_schema=CartState,
    checkpointer=InMemorySaver(),
    system_prompt="""شما یک دستیار فروشگاه الکترونیک هستید.
محصولات موجود: لپ‌تاپ، موبایل، هدفون، کیبورد، ماوس، وب‌کم
برای افزودن محصول از add_to_cart، برای مشاهده از show_cart، برای پاک کردن از clear_cart استفاده کنید.
همیشه به فارسی پاسخ بدهید."""
)

cart_config = {"configurable": {"thread_id": "cart-session-1"}}
print("Agent فروشگاه آماده است!")

Agent فروشگاه آماده است!


In [63]:
# تست خرید
resp = cart_agent.invoke(
    {"messages": [{"role": "user", "content": "یه لپ‌تاپ بذار تو سبدم"}]},
    config=cart_config
)
print(resp["messages"][-1].content)

لپ‌تاپ به سبد خرید شما اضافه شد. جمع کل سبد خرید شما اکنون ۴۵,۰۰۰,۰۰۰ تومان است. آیا کار دیگری می‌خواهید انجام دهید؟


In [65]:
# تست خرید
resp = cart_agent.invoke(
    {"messages": [{"role": "user", "content": "یه هدفون هم میخواهم"}]},
    config=cart_config
)
print(resp["messages"][-1].content)

هدفون به سبد خرید شما اضافه شد. جمع کل سبد خرید شما اکنون ۴۸,۵۰۰,۰۰۰ تومان است. آیا کار دیگری می‌خواهید انجام دهید؟


In [67]:
resp = cart_agent.invoke(
    {"messages": [{"role": "user", "content": "سبدمو نشون بده"}]},
    config=cart_config
)
print(resp["messages"][-1].content)

سبد خرید شما به شرح زیر است:

1. لپ‌تاپ: ۴۵,۰۰۰,۰۰۰ تومان
2. هدفون: ۳,۵۰۰,۰۰۰ تومان

جمع کل: ۴۸,۵۰۰,۰۰۰ تومان

آیا کار دیگری می‌خواهید انجام دهید؟


In [ ]:
# مشاهده مستقیم State
state = cart_agent.get_state(cart_config)
print("=== State مستقیم ===")
print(f"آیتم‌ها: {state.values.get('cart_items', [])}")
print(f"جمع کل: {state.values.get('total_price', 0):,.0f} تومان")

## ۵. آپدیت مستقیم State (بدون Agent)

می‌توانید مستقیماً State را از خارج آپدیت کنید:

In [ ]:
from langchain.messages import HumanMessage

# آپدیت مستقیم State بدون اینکه agent تصمیم بگیرد
cart_agent.update_state(
    cart_config,
    {"customer_name": "علی رضایی",
     "messages": [HumanMessage(content="[سیستم: مشتری احراز هویت شد]")]}
)

# مشاهده State آپدیت‌شده
state = cart_agent.get_state(cart_config)
print(f"اسم مشتری: {state.values.get('customer_name', 'تنظیم نشده')}")
print(f"تعداد پیام‌ها: {len(state.values['messages'])}")